# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def window(x):
    return src.utils.get_windowed(x, stride=120)


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### most data

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### "wide" subsurface data

In [ ]:
## should we use "wide" data?
USE_WIDE = True

if USE_WIDE:

    ## load spatial data
    forced_wide, anom_wide = load_consolidated_wide()

    for v in list(forced_wide):
        forced[v] = forced_wide[v]
        anom[v] = anom_wide[v]

#### max grad thermocline

In [ ]:
h_mg_forced, h_mg = src.utils.load_h_data(max_grad=True)

### scaling for mean thermocline depth

#### Mean thermocline depth

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

## Compute/plot

### Specify variabes

In [ ]:
## should we use mixed layer T?
# use_T_ml = True
use_T_ml = False

## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
# LONS_W = dict(longitude=slice(120, 160))
LONS_W = dict(longitude=slice(120, 180))
LONS_TAU = dict(longitude=slice(150, 230))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
TAU_FN_3 = lambda x: sel_helper(x, LATS, LONS_E)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)


## specify entrainment / ML averages
LON_AVG = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")
ENT_AVG = lambda x: x.sel(z_t=slice(50, 80)).mean("z_t")
ML_AVG = lambda x: x.sel(z_t=slice(None, 50)).mean("z_t")

## get T3 volume avg
T3_ENT_AVG = lambda x: ENT_AVG(LON_AVG(x))
T3_ML_AVG = lambda x: ML_AVG(LON_AVG(x))

if use_T_ml:
    anom["T_3"] = src.utils.reconstruct_wrapper(
        anom[["T", "T_comp"]],
        fn=T3_ML_AVG,
    )["T"]

In [ ]:
## should we use OHC
USE_OHC = True

lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

if USE_OHC:

    ## compute ohc
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W),
    )["T"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E),
    )["T"]

else:
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom_wide[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_W)),
    )["T"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_E)),
    )["T"]

#### helper funcs

In [ ]:
def regress_over_time(data, x_vars, y_vars, dims=["time", "member"]):
    """regression over time"""

    ## get windowed data
    data_ = window(data)

    ## empty list to hold coefficients
    coefs = []

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars, dims=dims)

    ## loop thru years
    for year in tqdm.tqdm(data_.year):

        ## get grouped data
        data_y = data_.sel(year=year).groupby("time.month")

        ## do regression
        coefs.append(data_y.map(src.utils.regress_xr_proj, **kwargs))

    return xr.concat(coefs, dim=data_.year)


def regress_wrapper(data, x_vars, y_var, y_fn, dims=["time", "member"]):
    """regression over time"""

    ## prep data
    y_data = src.utils.reconstruct_wrapper(data[[f"{y_var}", f"{y_var}_comp"]], fn=y_fn)

    ## subset for data
    data_ = xr.merge([data[x_vars], y_data])

    return regress_over_time(data_, x_vars=x_vars, y_vars=[y_var], dims=dims)


def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1


def make_scatter_ax(ax, anom_, xvar, yvar, month):
    """scatter plot of data for given month"""

    ## prep func
    get_season = lambda x: src.utils.sel_month(
        x.resample({"time": "QS-DEC"}).mean(), month
    )
    prep = lambda x: get_season(x).transpose("time", "member")

    ## get plot data
    plot_data = (prep(anom_[xvar]), prep(anom_[yvar]))

    ## get stats
    corr = xr.corr(*plot_data)
    cov = xr.cov(*plot_data)
    m = cov / plot_data[0].var()

    ## plot data
    ax.scatter(*plot_data, s=3, label=f"m = {m.item():.1e}")

    ax.set_title(f"corr = {corr.item():.2f}")

    ## formatting
    ax_kwargs = dict(ls="--", c="gray", lw=0.5)
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    ax.legend(prop=dict(size=10))

    return ax


def make_scatter(anom_, xvar, yvar, month):
    """scatter plot of data for given month"""

    ## prep func
    get_season = lambda x: src.utils.sel_month(
        x.resample({"time": "QS-DEC"}).mean(), month
    )
    prep = lambda x: get_season(x).transpose("time", "member")

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5), layout="constrained")

    for ax, t_idx in zip(axs, [["1850", "1889"], ["1990", "2029"], ["2061", "2100"]]):

        ## helper func
        prep_ = lambda x: prep(x).sel(time=slice(*t_idx))

        ## scatter plot of data
        ax = make_scatter_ax(
            ax,
            prep_(anom_),
            xvar=xvar,
            yvar=yvar,
            month=month,
        )

    ## format/scale axes
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    axs[2].set_yticks([])

    return fig, axs

#### specify how to compute

In [ ]:
BY_MEMBER = True

if BY_MEMBER:
    DIMS = ["time"]
else:
    DIMS = ["time", "member"]

### Surface terms

#### SST-$\tau_x$

In [ ]:
## compute indices
taux_w = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)["taux"]
taux_e = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN_3)["taux"]

## regress over time
anom_ = xr.merge([taux_w.rename("taux_w"), taux_e.rename("taux_e"), anom["T_3"]])
mu = regress_over_time(
    anom_, x_vars=["T_3"], y_vars=["taux_w", "taux_e"], dims=DIMS
).squeeze(drop=True)

## get coefs
mu_a = mu["taux_w"]
mu_a_prime = mu["taux_e"]

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="T_3", month=6)

for v in ["taux_w", "taux_e"]:
    print(v)
    fig, axs = make_scatter(yvar=v, **kwargs)
    plt.show()

#### $\tau_x$-SSH

In [ ]:
## get tau index
taux_wpac = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)

## get delta_x(h)
dh = (anom["h_e"] - anom["h_w"]).rename("dh")

## merge data
anom_ = xr.merge([taux_wpac, anom[["h_e", "h_w"]], dh])

## regress
beta_h = regress_over_time(anom_, x_vars=["taux"], y_vars=["dh"], dims=DIMS)
beta_h = beta_h["dh"].squeeze(drop=True)

In [ ]:
for yvar in ["dh", "h_e", "h_w"]:
    print(yvar)

    ## specify kwargs
    kwargs = dict(anom_=anom_, xvar="taux", yvar=yvar, month=3)
    fig, axs = make_scatter(**kwargs)
    plt.show()

#### $\tau_x$-$h$

In [ ]:
## get tau index
taux_wpac = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)

## get gradient of h
Hw_FN_mg = lambda x: sel_helper(x, LATS_H, dict(longitude=slice(120, 180)))
get_dh_mg = lambda x: He_FN(x) - Hw_FN_mg(x)

## get indices
anom_ = xr.merge([taux_wpac, get_dh_mg(h_mg).rename("dh")])


## regress (v1)
beta_h_mg = regress_over_time(data=anom_, x_vars=["taux"], y_vars=["dh"], dims=DIMS)
beta_h_mg = beta_h_mg["dh"].squeeze(drop=True)

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="taux", yvar="dh", month=3)
fig, axs = make_scatter(**kwargs)
plt.show()

### velocity terms

In [ ]:
## get tau func
taux_w = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)["taux"]
taux_e = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN_3)["taux"]

## get U, w in epac
U_epac = src.utils.reconstruct_wrapper(anom[["u", "u_comp"]], fn=T3_ML_AVG)
W_epac = src.utils.reconstruct_wrapper(anom[["w", "w_comp"]], fn=T3_ENT_AVG)

#### $\beta$

In [ ]:
## get data subset
anom_ = xr.merge(
    [anom["h_w"], U_epac, W_epac, taux_w.rename("taux_w"), taux_e.rename("taux_e")]
)

## do regression
beta = regress_over_time(
    data=anom_,
    x_vars=["taux_w", "taux_e", "h_w"],
    y_vars=["u", "w"],
    dims=DIMS,
)

# ## split by variable
beta_ur = beta["u"].sel(j="taux_w")
beta_ul = beta["u"].sel(j="taux_e")
beta_uh = beta["u"].sel(j="h_w")
beta_wr = -beta["w"].sel(j="taux_w")
beta_wl = -beta["w"].sel(j="taux_e")
beta_wh = beta["w"].sel(j="h_w")

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, yvar="w", month=12)
for xvar in ["taux_w", "taux_e", "h_w"]:
    print(xvar)
    fig, axs = make_scatter(xvar=xvar, **kwargs)
    plt.show()

### subsurface terms

#### SSH-$T_{sub}$

In [ ]:
## get ssh index
Tsub = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ENT_AVG)

## merge data
anom_ = xr.merge([anom["h_e"], Tsub])

## regress
a_h = regress_over_time(data=anom_, x_vars=["h_e"], y_vars=["T"], dims=DIMS)
a_h = a_h["T"].squeeze(drop=True).rename("a_h")

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="h_e", yvar="T", month=12)
fig, axs = make_scatter(**kwargs)
plt.show()

#### $h-T_{sub}$

In [ ]:
## get ssh and T indices
h_epac = He_FN(h_mg)
h_wpac = Hw_FN_mg(h_mg)
Tsub = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ENT_AVG)
anom_ = xr.merge([h_epac.rename("h_e"), h_wpac.rename("h_w"), Tsub])

## regress
a_h_mg = regress_over_time(data=anom_, x_vars=["h_e"], y_vars=["T"], dims=DIMS)
a_h_mg = a_h_mg["T"].squeeze(drop=True).rename("a_h")

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="h_e", yvar="T", month=12)
fig, axs = make_scatter(**kwargs)
plt.show()

#### $\overline{w}$

In [ ]:
## specify mixed layer depth
H0 = 50

## get w_bar
w_bar = src.utils.reconstruct_clim(window(forced[["w", "w_comp"]]))["w"]
is_downwelling = w_bar < 0

## only keep upwelling
w_bar_upwelling = w_bar.where(~(is_downwelling), other=0.0)

## average over Niño 3 and scale by MLD
w_H = 1 / H0 * T3_ENT_AVG(w_bar_upwelling).rename("w_H")

In [ ]:
sel = lambda x: x.mean(["month", "year"])

## plot w to make sure there's no downwelling
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(w_bar_upwelling.longitude, sel(ENT_AVG(w_bar_upwelling)))
ax.plot(w_bar_upwelling.longitude, sel(w_bar_upwelling.sel(z_t=50, method="nearest")))
ax.axhline(0, lw=0.8, c="gray")
plt.show()

#### $\frac{\partial T}{\partial z}$

In [ ]:
## specify mixed layer depth
H0 = 50

## get Tbar
T_bar = src.utils.reconstruct_clim(window(forced[["T", "T_comp"]]))["T"]

## get dTbar_dz
dTdz_bar = 1 / H0 * (ENT_AVG(T_bar) - ML_AVG(T_bar))
dTdz_bar_v2 = ML_AVG(T_bar.differentiate("z_t"))

## only keep upwelling
is_downwelling = ENT_AVG(w_bar) < 0
dTdz_bar_upwelling = dTdz_bar.where(~(is_downwelling), other=0.0)
dTdz_bar_upwelling_v2 = dTdz_bar_v2.where(~(is_downwelling), other=0.0)

## average over Niño 3
dTdz_bar_T3 = LON_AVG(dTdz_bar_upwelling).rename("dTdz_bar")

In [ ]:
sel = lambda x: x.mean(["year", "month"])

fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(dTdz_bar.longitude, sel(dTdz_bar), label=r"$\Delta_z T$")
ax.plot(dTdz_bar.longitude, sel(dTdz_bar_v2), label=r"$\frac{\partial T}{\partial z}$")

ax_kwargs = dict(c="gray", lw=0.8, ls="--")
ax.axvline(210, **ax_kwargs)
ax.axvline(270, **ax_kwargs)
ax.axhline(0, **ax_kwargs)
ax.legend()
plt.show()

#### damping

In [ ]:
## compute nhf (W/m2)
nhf = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3)[
    "nhf"
]

## convert to damping rate (units of K/mo)
sec_per_mo = 8.64e4 * 30
rho = 1.02e3
Cp = 4.2e3
Q = nhf * sec_per_mo / (rho * Cp * H0)

In [ ]:
## regress over time
anom_ = xr.merge([Q.rename("Q"), anom[["T_3", "T_34"]]])
alpha = regress_over_time(anom_, x_vars=["T_3"], y_vars=["Q"], dims=DIMS)
alpha = alpha["Q"].squeeze(drop=True)

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="T_3", yvar="Q", month=12)

fig, axs = make_scatter(**kwargs)
plt.show()

#### Mixing efficiency

In [ ]:
## compute gamma (following Jin et al, 2020)
gamma = 0.75 * xr.ones_like(beta_h)

#### Dynamical damping

In [ ]:
dd = -w_H * gamma

### Consolidate data

#### thermocline feedback

In [ ]:
## should we use ssh or thermocline based h?
use_ssh = True

if use_ssh:
    beta_h_plot = beta_h
    a_h_plot = a_h
    # scale = hbar_scale
    scale = 1

else:
    beta_h_plot = beta_h_mg
    a_h_plot = a_h_mg
    scale = 1

## apply scaling
beta_h_plot = (beta_h_plot * scale).rename("beta_h")
a_h_plot = (a_h_plot / scale).rename("a_h")

## compute THF
THF = (mu_a * beta_h_plot * a_h_plot * gamma * w_H).rename("thf")

## put data in xarray
params_thf = xr.merge([THF, mu_a.rename("mu_a"), beta_h_plot, a_h_plot, w_H])
labels_thf = [r"THF", r"$\mu_a$", r"$\beta_h$", r"$a_h$", r"$w/H$"]
label_dict_thf = dict(zip(list(params_thf), labels_thf))

#### Ekman feedback

In [ ]:
## compute EKF
EKF = -((mu_a * beta_wr + mu_a_prime * beta_wl) * dTdz_bar_T3).rename("ekf")

## put data in xarray
params_ekf = xr.merge(
    [
        EKF,
        mu_a.rename("mu_a"),
        mu_a_prime.rename("mu_a_prime"),
        beta_wr.rename("beta_wr"),
        beta_wl.rename("beta_wl"),
        dTdz_bar_T3,
    ],
    compat="override",
)
labels_ekf = [
    r"EKM",
    r"$\mu_a$",
    r"$\mu_a'$",
    r"$\beta_{wr}$",
    r"$\beta_{wl}$",
    r"$\partial T / \partial z$",
]
label_dict_ekf = dict(zip(list(params_ekf), labels_ekf))

#### Bulk estimates

Thermocline

In [ ]:
## get ssh index
Tsub = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ENT_AVG)["T"]
# dh_tilde = (Tsub.groupby("time.month") / params_thf["a_h"]) - h_w
dh_tilde = anom["h_e"] - anom["h_w"]

## merge data
anom_ = xr.merge([dh_tilde.rename("dh"), anom["h_w"], anom["T_3"], Tsub.rename("Tsub")])

## regress
thf_coefs_bulk = regress_over_time(data=anom_, x_vars=["T_3"], y_vars=["dh"], dims=DIMS)
# thf_coefs_bulk = regress_over_time(data=anom_, x_vars=["T_3", "h_w"], y_vars=["Tsub"])

# ## split up
scale = params_thf["w_H"] * params_thf["a_h"] * gamma
R_thf_bulk = scale * thf_coefs_bulk["dh"].sel(j="T_3")
# R_thf_bulk = params_thf["w_H"] * thf_coefs_bulk["Tsub"].sel(j="T_3")

# ## add dynamical damping
# R_thf_bulk = R_thf_bulk + dd

Ekman

In [ ]:
## get ssh index
w = src.utils.reconstruct_wrapper(anom[["w", "w_comp"]], fn=T3_ENT_AVG)["w"]

## merge data
anom_ = xr.merge([anom["h_w"], w, anom["T_3"]])

## regress
ekf_coefs_bulk = regress_over_time(
    data=anom_, x_vars=["T_3", "h_w"], y_vars=["w"], dims=DIMS
)

## split up
R_ekf_bulk = params_ekf["dTdz_bar"] * ekf_coefs_bulk["w"].sel(j="T_3")
F1_ekf_bulk = params_ekf["dTdz_bar"] * ekf_coefs_bulk["w"].sel(j="h_w")

Sum to get total

In [ ]:
R_bulk = R_thf_bulk + R_ekf_bulk + alpha + dd
R_phys = params_thf["thf"] + params_ekf["ekf"] + alpha + dd

Seasonal cycles

In [ ]:
## specify years to plot
YEARS = [1870, 2020, 2090]
# YEARS = [1870, 1960]

for p_phys, p_bulk, name in zip(
    [R_phys, params_thf["thf"], params_ekf["ekf"]],
    [R_bulk, R_thf_bulk, R_ekf_bulk],
    ["R", "THF", "EKF"],
):

    ## make plot
    fig, axs = plt.subplots(1, 2, figsize=(4.5, 2), layout="constrained")

    for ax, p, label in zip(
        axs,
        [p_phys, p_bulk],
        ["physics", "bulk"],
    ):
        ax.set_title(f"{name} ({label})")
        ax.axhline(0, ls="--", c="gray", lw=0.8)
        for year in YEARS:
            # for year in [1870, 2020, 2090]:
            ax.plot(p.month, p.sel(year=year).mean("member"))

    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    plt.show()

Change over time

In [ ]:
## specify month range
MONTH_RANGE = (1, 12)
# MONTH_RANGE = (2, 5)


## choose how to select data
diff = lambda x: x - x.isel(year=0)
sel = lambda x: diff(x).sel(month=slice(*MONTH_RANGE)).mean("month")
sel_mean = lambda x: sel(x).mean("member")


## make plot
fig, axs = plt.subplots(1, 4, figsize=(10, 2), layout="constrained")
for ax, p_phys, p_bulk, name in zip(
    axs,
    [R_phys, params_thf["thf"] + dd, params_ekf["ekf"], alpha],
    [R_bulk, R_thf_bulk + dd, R_ekf_bulk, alpha],
    ["R", "THF+DD", "EKF", r"$\alpha$"],
):

    ## plot total
    ax.plot(p_phys.year, sel_mean(p_phys), label="physics")
    ax.plot(p_bulk.year, sel_mean(p_bulk), label="bulk")

    ## format/labeling
    ax_kwargs = dict(c="k", lw=0.8, ls="--")
    ax.axhline(0, **ax_kwargs)
    ax.set_xticks([1870, 1960, 2020])
    for t in ax.get_xticks()[[0, -1]]:
        ax.axvline(t, **ax_kwargs)
    ax.legend()
    ax.set_title(name)

src.utils.set_lims(axs)
plt.show()

#### ZAF feedback

In [ ]:
## to-do...

### Validate
Compare predictions of $T_{sub}$

In [ ]:
## specify year to look at
YEAR = 1870

## get data
params_ = params_thf.sel(year=YEAR)
Tsub = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ENT_AVG)["T"]
Tml = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ML_AVG)["T"]
data_ = window(
    xr.merge(
        [
            Tsub.rename("Tsub"),
            anom[["h_w", "h_e"]],
            (anom["h_e"] - anom["h_w"]).rename("dh"),
            anom["T_3"],
        ]
    )
).sel(year=YEAR)

## forecast using physics
dh_hat = data_["T_3"].groupby("time.month") * (params_["mu_a"] * params_["beta_h"])
he_hat = data_["h_w"] + dh_hat
Tsub_hat = he_hat.groupby("time.month") * params_["a_h"]

## forecast using bulk
thf_coefs_bulk_ = thf_coefs_bulk["dh"].sel(j="T_3", year=YEAR)
dh_hat_bulk = data_["T_3"].groupby("time.month") * thf_coefs_bulk_
he_hat_bulk = data_["h_w"] + dh_hat_bulk
Tsub_hat_bulk = he_hat_bulk.groupby("time.month") * params_["a_h"]

## merge results
res = xr.merge(
    [
        data_,
        Tsub_hat.rename("Tsub_hat"),
        dh_hat.rename("dh_hat"),
        he_hat.rename("he_hat"),
        Tsub_hat_bulk.rename("Tsub_hat_bulk"),
        dh_hat_bulk.rename("dh_hat_bulk"),
        he_hat_bulk.rename("he_hat_bulk"),
    ]
)

##### $h_e$

In [ ]:
MONTH = 12

fig, axs = plt.subplots(1, 2, figsize=(5, 2.5), layout="constrained")

## shared args for plots
kwargs = dict(anom_=res, month=MONTH)


make_scatter_ax(ax=axs[0], xvar="h_e", yvar="he_hat", **kwargs)
make_scatter_ax(ax=axs[1], xvar="h_e", yvar="he_hat_bulk", **kwargs)


for ax, amp in zip(axs, [500, 500]):
    # ax.set_aspect("equal")
    zz = np.linspace(-amp, amp)
    ax.plot(zz, zz, c="k", lw=1)

plt.show()

##### $\Delta h$

In [ ]:
MONTH = 12

fig, axs = plt.subplots(1, 2, figsize=(5, 2.5), layout="constrained")

## shared args for plots
kwargs = dict(anom_=res, month=MONTH)


make_scatter_ax(ax=axs[0], xvar="dh", yvar="dh_hat", **kwargs)
make_scatter_ax(ax=axs[1], xvar="dh", yvar="dh_hat_bulk", **kwargs)


for ax, amp in zip(axs, [500, 500]):
    # ax.set_aspect("equal")
    zz = np.linspace(-amp, amp)
    ax.plot(zz, zz, c="k", lw=1)

plt.show()

#### $T_{sub}$

In [ ]:
MONTH = 12

fig, axs = plt.subplots(1, 2, figsize=(5, 2.5), layout="constrained")

## shared args for plots
kwargs = dict(anom_=res, month=MONTH)


make_scatter_ax(ax=axs[0], xvar="Tsub", yvar="Tsub_hat", **kwargs)
make_scatter_ax(ax=axs[1], xvar="Tsub", yvar="Tsub_hat_bulk", **kwargs)


for ax, amp in zip(axs, [3, 3]):
    # ax.set_aspect("equal")
    zz = np.linspace(-amp, amp)
    ax.plot(zz, zz, c="k", lw=1)

plt.show()

### Plots

In [ ]:
def plot_annual_mean(params, label_dict, month_range=(1, 12)):
    """plot annual mean parameters and components"""

    ## get difference
    delta_p = frac_change(params.sel(month=slice(*month_range)).mean("month"))

    ## get name of first parameter
    p0 = list(params)[0]

    ## make plot
    fig, axs = plt.subplots(1, 2, figsize=(6, 2.75), layout="constrained")

    ## plot total
    axs[0].plot(params.year, delta_p[p0], c="k", label=label_dict[p0])

    # plot terms
    for p in list(params)[1:]:
        axs[1].plot(params.year, delta_p[p], label=label_dict[p])

    ## format/labeling
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    ax_kwargs = dict(c="k", lw=0.8, ls="--")
    for ax in axs:
        ax.axhline(0, **ax_kwargs)
        ax.axvline(2020, **ax_kwargs)
        ax.set_xticks([1870, 2020])
        ax.legend()

    return fig, axs


def plot_seasonal_cycle(params, label_dict, years=[1870, 2020, 2090]):
    """plot snapshots of parameters' seasonal cycle"""

    ## count number of parameters
    n = len(list(params))

    fig, axs = plt.subplots(1, n, figsize=(n * 2, 2), layout="constrained")

    for ax, p in zip(axs, list(params)):

        ax.set_title(label_dict[p])
        for year in years:
            ax.plot(params.month, params[p].sel(year=year))

        ## formatting
        ax.axvspan(1.5, 5.5, alpha=0.1, color="gray", linewidth=0)
        ax.axhline(0, ls="--", c="k", lw=0.8)

    return fig, ax

#### Total

Annual mean

In [ ]:
## specify month range
# MONTH_RANGE = (1, 12)
MONTH_RANGE = (2, 5)

## choose how to select data
diff = lambda x: x - x.isel(year=0)
sel = lambda x: diff(x).sel(month=slice(*MONTH_RANGE)).mean(["month", "member"])

## get sum
R = params_thf["thf"] + params_ekf["ekf"] + alpha

## make plot
fig, ax = plt.subplots(figsize=(3, 2.75), layout="constrained")

## plot total
ax.plot(params_thf.year, sel(params_thf["thf"]), label="THF")
ax.plot(params_ekf.year, sel(params_ekf["ekf"]), label="EKF")
ax.plot(params_ekf.year, sel(alpha), label=r"$\alpha$")
ax.plot(params_ekf.year, sel(R), label=r"$R$", c="k")

## format/labeling
ax_kwargs = dict(c="k", lw=0.8, ls="--")
ax.axhline(0, **ax_kwargs)
ax.set_xticks([1870, 1960, 2020])
for t in ax.get_xticks()[[0, -1]]:
    ax.axvline(t, **ax_kwargs)
ax.legend()

plt.show()

Seasonal cycle

In [ ]:
## specify years to plot
YEARS = [1870, 2020, 2090]
# YEARS = [1870, 1960]

## make plot
fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

for ax, p, label in zip(
    axs,
    [R, params_thf["thf"], params_ekf["ekf"], -alpha],
    [r"$R$", "THF", "EKF", r"-$\alpha$"],
):
    ax.set_title(label)
    ax.axhline(0, ls="--", c="gray", lw=0.8)
    for year in YEARS:
        # for year in [1870, 2020, 2090]:
        ax.plot(p.month, p.sel(year=year).mean("member"))

src.utils.set_lims(axs)
plt.show()

In [ ]:
## compare thermocline feedback to estimate from Jin et al (2020)
print(f"{7.53e-3 * 1383.9 * 0.138 * 7.62e-8 * sec_per_mo:.2f}")
print(f"{(mu_a.mean() * beta_h.mean() * a_h.mean() * w_H.mean()).item():.2f}")

#### THF

In [ ]:
def plot_quantiles(ax, x, label=None, month=None, time_dim="time", **kwargs):
    """plot .1, .5, and .9 quantiles on given ax object"""

    ## compute quantiles
    x_quantiles = x.quantile([0.1, 0.5, 0.9], dim="member")

    ## plot median
    ax.plot(
        x_quantiles[time_dim],
        x_quantiles.sel(quantile=0.5),
        lw=2,
        label=label,
        **kwargs,
    )

    ## plot upper/lower bounds
    for q in [0.1, 0.9]:
        ax.plot(
            x_quantiles[time_dim],
            x_quantiles.sel(quantile=q),
            lw=1,
            alpha=0.5,
            **kwargs,
        )

    return

In [ ]:
fig, axs = plot_annual_mean(params_thf.mean("member"), label_dict_thf)
plt.show()

fig, axs = plot_annual_mean(
    params_thf.mean("member"), label_dict_thf, month_range=[4, 5]
)
plt.show()


## specify years to plot for seasonal cycle
# years = [1870, 1960]
years = [1870, 2020, 2090]

## make plot
fig, axs = plot_seasonal_cycle(params_thf.mean("member"), label_dict_thf, years=years)
plt.show()

In [ ]:
def frac_change_ens(x):
    """fractional change for ensemble"""
    x0 = x.isel(year=0).sel(quantile=0.5)

    return x / x0 - 1

In [ ]:
## get params to plot
# plot_params = xr.merge([params_thf, alpha.rename("alpha")]).mean("month")
plot_params = (
    xr.merge([params_thf, alpha.rename("alpha")]).sel(month=[5, 6, 7]).mean("month")
)
stats = plot_params.quantile(q=[0.1, 0.5, 0.9], dim="member")
stats_change = frac_change_ens(stats)

fig, axs = plt.subplots(1, 5, figsize=(10.5, 2), layout="constrained")

for ax, param_ in zip(axs, ["thf", "mu_a", "beta_h", "a_h", "alpha"]):

    for q, lw in zip([0.5, 0.1, 0.9], [2, 1, 1]):

        ax.plot(
            # plot_params.year, plot_params[param_].quantile(q=q, dim="member"), lw=lw,c="k",
            plot_params.year,
            stats_change[param_].sel(quantile=q),
            lw=lw,
            c="k",
        )
    ax.axhline(0, ls="--", c="gray", lw=0.8)
    ax.set_title(param_)

src.utils.set_lims(axs[:-1])

In [ ]:
prod = params_thf["beta_h"] * params_thf["a_h"] * params_thf["w_H"]

delta = lambda x: x / x.isel(year=0) - 1
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(params_thf.year, delta(prod.mean(["month", "member"])))
ax.set_ylim([-0.05, 0.08])
ax.axhline(0, c="k")
plt.show()

#### EKF

In [ ]:
fig, axs = plot_annual_mean(params_ekf.mean("member"), label_dict_ekf)
axs[1].set_ylim([None, 0.75])
plt.show()

# fig,axs = plot_annual_mean(params_thf, label_dict_thf, month_range=[2,5])
# plt.show()

fig, axs = plot_seasonal_cycle(params_ekf.mean("member"), label_dict_ekf)
plt.show()

In [ ]:
## get params to plot
plot_params = params_ekf.mean("month")
stats = plot_params.quantile(q=[0.1, 0.5, 0.9], dim="member")
stats_change = frac_change_ens(stats)

fig, axs = plt.subplots(1, 5, figsize=(10.5, 2), layout="constrained")

for ax, param_ in zip(axs, list(plot_params)):

    for q, lw in zip([0.5, 0.1, 0.9], [2, 1, 1]):

        ax.plot(
            # plot_params.year, plot_params[param_].quantile(q=q, dim="member"), lw=lw,c="k",
            plot_params.year,
            stats_change[param_].sel(quantile=q),
            lw=lw,
            c="k",
        )
    ax.axhline(0, ls="--", c="gray", lw=0.8)
    ax.set_title(param_)

src.utils.set_lims(axs[:-1])